##### *Copyright 2020 Google LLC*
*Licensed under the Apache License, Version 2.0 (the "License")*

In [ ]:
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Retrain a detection model for Edge TPU with quant-aware training (TF 1.15)

This notebook uses a set of TensorFlow training scripts to perform transfer-learning on a quantization-aware object detection model and then convert it for compatibility with the [Edge TPU](https://coral.ai/products/).

Specifically, this tutorial shows you how to retrain a MobileNet V1 SSD model so that it detects two pets: Abyssinian cats and American Bulldogs (from the [Oxford-IIIT Pets Dataset](https://www.robots.ox.ac.uk/~vgg/data/pets/)), using TensorFlow r1.15.

Beware that, compared to a desktop computer, this training can take *a lot* longer in Colab because Colab provides limited resources for long-running operations. So you'll likely see faster training speeds if you [connect this notebook to a local runtime](https://research.google.com/colaboratory/local-runtimes.html), or instead follow the [tutorial to run this training in Docker](https://coral.ai/docs/edgetpu/retrain-detection/) (which includes more documentation about this process).

## Import TensorFlow

In [ ]:
import os

# TensorFlow 1.x is no longer supported in Colab, so these commands are removed.
# Colab defaults to TensorFlow 2.x.

! pip install -U pycocotools

In [ ]:
import tensorflow as tf
print(tf.__version__)

## Clone the model and training repos

In [ ]:
! git clone https://github.com/tensorflow/models.git

In [ ]:
# The previous git checkout was for a very old TF1.x version of models.
# Switching to master branch for TF2 compatibility.
! cd models && git checkout master

In [ ]:
! git clone https://github.com/google-coral/tutorials.git

! cp -r tutorials/docker/object_detection/scripts/* models/research/

## Import dependencies

For details, see https://github.com/tensorflow/models/blob/master/research/object_detection/g3doc/installation.md

In [ ]:
# 'python' package is deprecated. Using 'python3-tk' and adding 'protobuf-compiler'
# for the TensorFlow 2.x Object Detection API requirements.
! apt-get install -y python3-tk protobuf-compiler
! pip install Cython contextlib2 pillow lxml jupyter matplotlib

In [ ]:
# Removed manual protoc download and install. Protobuf compiler is installed via apt-get.
# However, the apt-get version is too old for the installed protobuf python library.
# We will download a newer protoc binary and use it to compile.

# Download protoc (e.g., 3.20.3)
! wget -q https://github.com/protocolbuffers/protobuf/releases/download/v3.20.3/protoc-3.20.3-linux-x86_64.zip
! unzip -o protoc-3.20.3-linux-x86_64.zip -d protoc_temp
! mv protoc_temp/bin/protoc /usr/local/bin/protoc_new
! chmod +x /usr/local/bin/protoc_new

# Now, compile protobufs with the new protoc and install the Object Detection API for TensorFlow 2.x.
! cd models/research && protoc_new object_detection/protos/*.proto --python_out=.
! cd models/research && cp object_detection/packages/tf2/setup.py .
! cd models/research && python -m pip install .

# Clean up
! rm -rf protoc_temp protoc-3.20.3-linux-x86_64.zip

In [ ]:
# Install pycocoapi
! git clone --depth 1 https://github.com/cocodataset/cocoapi.git
! (cd cocoapi/PythonAPI && make -j8)
! cp -r cocoapi/PythonAPI/pycocotools/ models/research/
! rm -rf cocoapi

In [ ]:
# This cell is now redundant as protoc compilation and API installation are handled in the previous cell.

In [ ]:
import os
os.environ['PYTHONPATH'] += ":/content/models/research:/content/models/research/slim"

Just to verify everything is correctly set up:

In [ ]:
! cd models/research && python object_detection/builders/model_builder_test.py

In [ ]:
# Patch freezable_sync_batch_norm.py to use BatchNormalization instead of SyncBatchNormalization
! sed -i "s/tf.keras.layers.SyncBatchNormalization/tf.keras.layers.BatchNormalization/g" models/research/object_detection/core/freezable_sync_batch_norm.py

In [ ]:
import subprocess

# Run the model_builder_test.py and capture its output and exit code
result = subprocess.run(['python', 'object_detection/builders/model_builder_test.py'], cwd='models/research', capture_output=True, text=True)

# Print the captured standard output and standard error
print(result.stdout)
print(result.stderr)

# Check the exit code
if result.returncode == 0:
    print("\nmodel_builder_test.py passed successfully!")
else:
    print(f"\nmodel_builder_test.py failed with exit code {result.returncode}.")

The next step is to prepare the dataset and convert it into the TFRecord format, which is required by the TensorFlow Object Detection API for training. The provided script `./prepare_checkpoint_and_dataset.sh` handles this automatically.

This script will:
1.  Download the specified dataset (e.g., Oxford-IIIT Pets).
2.  Convert the images and annotations into TFRecord files (e.g., `pet_train.record` and `pet_val.record`).
3.  Download a pre-trained model checkpoint to be used for transfer learning.

You can examine cell `Mbz5nKlDAorQ` for the command that executes this script:

## Convert training data to TFRecord

To train with different images, read [how to configure your own training data](https://coral.ai/docs/edgetpu/retrain-detection/#configure-your-own-training-data).

In [ ]:
! echo "Current directory before operations: $(pwd)" && \
  if [ ! -d "/content/models" ]; then \
    echo "/content/models not found. Cloning tensorflow/models..."; \
    git clone https://github.com/tensorflow/models.git; \
  fi && \
  cd /content/models && git checkout master > /dev/null && cd /content && \
  if [ ! -d "/content/tutorials" ]; then \
    echo "/content/tutorials not found. Cloning google-coral/tutorials..."; \
    git clone https://github.com/google-coral/tutorials.git; \
  fi && \
  cp -r tutorials/docker/object_detection/scripts/* models/research/ && \
  cd /content/models/research && \
  echo "Current working directory after cd: $(pwd)" && \
  echo "\nOriginal 'source constants.sh' line:" && \
  grep "source .*constants.sh" prepare_checkpoint_and_dataset.sh && \
  sed -i 's|source /content/constants.sh|source ./constants.sh|g' prepare_checkpoint_and_dataset.sh && \
  echo "\nPatched 'source constants.sh' line:" && \
  grep "source .*constants.sh" prepare_checkpoint_and_dataset.sh && \
  ./prepare_checkpoint_and_dataset.sh --network_type mobilenet_v1_ssd --train_whole_model false

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

script_path = 'models/research/prepare_checkpoint_and_dataset.sh'
if os.path.exists(script_path):
    print(f"The script '{script_path}' exists.")
else:
    print(f"Error: The script '{script_path}' does not exist. Please ensure it was copied correctly.")


## Perform transfer-learning

The following script takes several hours to finish in Colab. (You can shorten by reducing the steps, but that reduces the final accuracy.)

If you didn't already select "Run all" then you should run all remaining cells now. That will ensure the rest of the notebook completes while you are away, avoiding the chance that the Colab runtime times-out and you lose the training data before you download the model.

In [ ]:
%env NUM_TRAINING_STEPS=500
%env NUM_EVAL_STEPS=100

# If you're retraining the whole model, we suggest thes values:
# %env NUM_TRAINING_STEPS=50000
# %env NUM_EVAL_STEPS=2000

In [ ]:
! ./retrain_detection_model.sh --num_training_steps $NUM_TRAINING_STEPS --num_eval_steps $NUM_EVAL_STEPS

As training progresses, you can see new checkpoint files appear in the `models/research/learn_pet/train/` directory.

## Compile for the Edge TPU

In [ ]:
! ./convert_checkpoint_to_edgetpu_tflite.sh --checkpoint_num $NUM_TRAINING_STEPS

In [ ]:
! curl https://packages.cloud.google.com/apt/doc/apt-key.gpg | sudo apt-key add -

! echo "deb https://packages.cloud.google.com/apt coral-edgetpu-stable main" | sudo tee /etc/apt/sources.list.d/coral-edgetpu.list

! sudo apt-get update

! sudo apt-get install edgetpu-compiler

In [ ]:
%cd learn_pet/models/

! ls

In [ ]:
! edgetpu_compiler output_tflite_graph.tflite

Download the files:

In [ ]:
from google.colab import files

files.download('output_tflite_graph_edgetpu.tflite')
files.download('labels.txt')

If you get a "Failed to fetch" error here, it's probably because the files weren't done saving. So just wait a moment and try again.

Also look out for a browser popup that might need approval to download the files.

## Run the model on the Edge TPU




You can now run the model on your Coral device with acceleration on the Edge TPU.

First, find some photos to try. Remember that you've trained this model to recognize just two classes: Abyssinian cats and
American Bulldogs. So here are a couple images that should provide results (provided by the
[Open Images Dataset](https://storage.googleapis.com/openimages/web/index.html)):

```
wget https://c4.staticflickr.com/8/7580/15865399370_ffa5b49d20_z.jpg -O dog.jpg && \
wget https://c6.staticflickr.com/9/8534/8652503705_687d957a29_z.jpg -O cat.jpg
```

Then, try running an inference using [this example code for the PyCoral API](https://github.com/google-coral/pycoral/blob/master/examples/detect_image.py). Just clone that repo and run the script using the model files you downloaded above (also be sure you have [installed the PyCoral API](https://coral.ai/software/#pycoral-api)):

```
git clone https://github.com/google-coral/pycoral

cd pycoral/examples/

python3 detect_image.py \
  --model output_tflite_graph_edgetpu.tflite \
  --labels labels.txt \
  --input dog.jpg \
  --output dog_result.jpg
```

Check out more examples for running inference at [coral.ai/examples](https://coral.ai/examples/#code-examples/).

## Implementation details



All the scripts used in this notebook come from the following locations:<br>
+  https://github.com/google-coral/tutorials/tree/master/docker/object_detection/scripts
+  https://github.com/tensorflow/models/tree/r1.13.0/research/object_detection/

More explanation of the steps in this tutorial is available at
https://coral.ai/docs/edgetpu/retrain-detection/.